# Data Preparation


In [88]:
import pandas as pd

# Load the Excel file into a dataframe
raw_df = pd.read_excel('lending_clubFull_Data_Set.xlsx')

### Features we will include for prediction
| Feature | Description |
| --- | --- |
| ```emp_length``` | employment length in years |
| ```home_ownership``` | whether borrower rents, owns, has mortgage, etc. |
| ```dti``` | monthly debt to monthly income |
| ```delinq_2yrs``` | number of 30+ dpd (days past due) delinquencies in the past 2 years |
| ```earliest_cr_line``` | month of the borrower's earliest reported credit line |
| ```inq_last_6mths``` | no. of inquiries in the past 6 months (except auto or mortgage) |
| ```revol_util``` | amt. of credit borrower is using relative to all available revolving credit |
| ```acc_now_delinq``` | current no. of delinquent accounts |
| ```bc_util``` | ratio of total current balance to high credit/credit limit for all bankcard accounts |
| ```num_tl_120dpd_2m``` | no. of accounts currently 120 dpd (updated in past 2 months) |
| ```num_tl_30dpd``` | no. of accounts currently 30 dpd (updated in past 2 months) |
| ```num_tl_90g_dpd_24m``` | no. of accounts 90+ dpd in last 24 months |

Target variable: ```loan_status```<br>
Possible values: Charged Off, Current, Fully Paid, Late, In Grace Period, Default

We convert this column to binary data, where the value is 1 if in default or charged off, and 0 otherwise.

In [184]:
# Keep only the selected features, plus the target variable
df = raw_df[[
    'loan_status',
    'emp_length',
    'home_ownership',
    'dti',
    'delinq_2yrs',
    'earliest_cr_line',
    'inq_last_6mths',
    'revol_util',
    'acc_now_delinq',
    'bc_util',
    'num_tl_120dpd_2m',
    'num_tl_30dpd',
    'num_tl_90g_dpd_24m'
]]

# Remove loans that do not meet the credit policy
df = df[~df['loan_status'].str.contains('Does not meet the credit policy. Status:', na=False)]

# convert loan_status column to binary values - 1 if default, charged off, or 30+ days late, otherwise 0
df['loan_status'] = df['loan_status'].apply(lambda x: 1 if x in ['Default', 'Charged Off', 'Late (31-120 days)'] else 0)

Now we clean the data for the other features.

In [186]:
# reduce home_ownership column to OWN, MORTGAGE, RENT
for keyword in ['ANY', 'OTHER', 'NONE']:
    df = df[~df['home_ownership'].str.contains(keyword, na=False)]

# convert home_ownership column to numbers
df['home_ownership'] = df['home_ownership'].replace('OWN', 1)
df['home_ownership'] = df['home_ownership'].replace('MORTGAGE', 0)
df['home_ownership'] = df['home_ownership'].replace('RENT', -1)

# convert emp_length column to numbers
df['emp_length'] = df['emp_length'].fillna(0)
df['emp_length'] = df['emp_length'].replace('< 1 year', 0.5)
df['emp_length'] = df['emp_length'].replace('1 year', 1)
for i in range(2, 10):
    find = str(i) + ' years'
    df['emp_length'] = df['emp_length'].replace(find, i)
df['emp_length'] = df['emp_length'].replace('10+ years', 10)
df['emp_length'] = pd.to_numeric(df['emp_length'], errors='coerce')

# remove rows which have empty cells in certain columns
for col in ['home_ownership', 'dti', 'revol_util', 'bc_util']:
    df = df.dropna(subset=[col])

df['num_tl_120dpd_2m'] = df['num_tl_120dpd_2m'].fillna(0)
df['num_tl_30dpd'] = df['num_tl_30dpd'].fillna(0)#
df['num_tl_90g_dpd_24m'] = df['num_tl_90g_dpd_24m'].fillna(0)

# convert earliest cr_line to numerical data
epoch = pd.Timestamp('1970-01-01')
df['earliest_cr_line'] = (df['earliest_cr_line'].dt.year - epoch.year) * 12 + (df['earliest_cr_line'].dt.month - epoch.month)

In [187]:
default = df['loan_status'].values

print(f"Observations: {len(df)}")
print(f"Default rate: {default.mean():.1%}")

Observations: 23968
Default rate: 11.5%


We will split the data in training (80%) and test (20%) sets, and standardize the training set features.

In [140]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('loan_status', axis=1)
y = df['loan_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline Classification
### 1. Logistic Regression

In [191]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

LR_THRESHOLD = 0.15

# Fit logistic regression
log_reg = LogisticRegression()
log_reg.fit(X_train_scaled, y_train)

# Get predictions on test set
y_pred = log_reg.predict(X_test_scaled)

# Predict probabilities and apply threshold
prob_pred = log_reg.predict_proba(X_test_scaled)[:, 1]
class_pred = (prob_pred > LR_THRESHOLD).astype(int)

print(f"Using threshold = {LR_THRESHOLD}:")
print(f"  LR Accuracy: {accuracy_score(y_test, class_pred):.4f}")

precision = precision_score(y_test, class_pred)
recall = recall_score(y_test, class_pred)
f1 = f1_score(y_test, class_pred)

print(f"  LR Precision: {precision:.3f}")
print(f"  LR Recall: {recall:.3f}")
print(f"  LR F1-Score: {f1:.3f}")

cm = confusion_matrix(y_test, class_pred)
print("Confusion Matrix:")
print(cm)

Using threshold = 0.15:
  LR Accuracy: 0.7847
  LR Precision: 0.171
  LR Recall: 0.247
  LR F1-Score: 0.202
Confusion Matrix:
[[3631  633]
 [ 399  131]]


### 2. Linear Discriminant Analysis

In [136]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

LDA_THRESHOLD = 0.15

# Create and fit LDA
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_scaled, y_train)

# Make predictions on test set
y_pred_lda = lda.predict(X_test_scaled)
prob_pred_lda = lda.predict_proba(X_test_scaled)[:, 1]
class_pred_lda = (prob_pred_lda > LDA_THRESHOLD).astype(int)

print(f"Using threshold = {LDA_THRESHOLD}:")
print(f"  LDA Accuracy: {accuracy_score(y_test, class_pred_lda):.4f}")
print(f"  LDA Precision: {precision_score(y_test, class_pred_lda):.4f}")
print(f"  LDA Recall: {recall_score(y_test, class_pred_lda):.4f}")
print(f"  LDA F1-Score: {f1_score(y_test, class_pred_lda):.4f}")

cm = confusion_matrix(y_test, class_pred_lda)
print("Confusion Matrix:")
print(cm)

Using threshold = 0.15:
  LDA Accuracy: 0.7801
  LDA Precision: 0.1684
  LDA Recall: 0.2509
  LDA F1-Score: 0.2015
Confusion Matrix:
[[3607  657]
 [ 397  133]]


### 3. _K_-Nearest Neighbours

In [142]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

k_values = range(1, 31)
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    cv_scores.append(scores.mean())

best_k = k_values[np.argmax(cv_scores)]
print(f"Best k: {best_k} with CV accuracy: {max(cv_scores):.3f}")

Best k: 24 with CV accuracy: 0.884


### 4. Decision Tree
Use cross-validation to find optimal pre-prune hyperparameters.

In [163]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

# Define hyperparameter grid to search
param_grid = {
    'max_depth': [3, 5, 7, 10, 15, 20],
    'min_samples_split': [2, 3, 5, 10, 20],
    'min_samples_leaf': [1, 2, 3, 4, 8]
}

# Create a decision tree classifier
dt_base = DecisionTreeClassifier(random_state=42)

# Perform grid search with cross-validation
grid_search = GridSearchCV(
    dt_base, 
    param_grid, 
    cv=5,  # 5-fold cross-validation
    scoring='f1',  # or 'accuracy', 'precision', 'recall', etc.
    n_jobs=-1  # Use all processors
)

# Fit the grid search
grid_search.fit(X_train_scaled, y_train)

# Print results
print(f"Best hyperparameters: {grid_search.best_params_}")
print(f"Best cross-validation F1 score: {grid_search.best_score_:.4f}")

# Train final model with best hyperparameters
dt_best = grid_search.best_estimator_

# Evaluate on test set
y_pred_dt_best = dt_best.predict(X_test_scaled)
print(f"\nTest Set Performance:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_dt_best):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_dt_best):.4f}")
print(f"  Recall: {recall_score(y_test, y_pred_dt_best):.4f}")
print(f"  F1-Score: {f1_score(y_test, y_pred_dt_best):.4f}")

cm_best = confusion_matrix(y_test, y_pred_dt_best)
print("Confusion Matrix:")
print(cm_best)

Best hyperparameters: {'max_depth': 20, 'min_samples_leaf': 3, 'min_samples_split': 2}
Best cross-validation F1 score: 0.1304

Test Set Performance:
  Accuracy: 0.8221
  Precision: 0.1387
  Recall: 0.1170
  F1-Score: 0.1269
Confusion Matrix:
[[3879  385]
 [ 468   62]]


In [173]:
# Create and fit Decision Tree
dt = DecisionTreeClassifier(max_depth=20, min_samples_leaf=3, min_samples_split=2, random_state=42)
dt.fit(X_train_scaled, y_train)

# Make predictions on test set
y_pred_dt = dt.predict(X_test_scaled)

# Evaluate
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Decision Tree Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Decision Tree Recall: {recall_score(y_test, y_pred_dt):.4f}")
print(f"Decision Tree F1-Score: {f1_score(y_test, y_pred_dt):.4f}")
print(f"Tree depth: {dt.get_depth()}")
print(f"Number of leaves: {dt.get_n_leaves()}")
print(f"Training accuracy: {dt.score(X_train_scaled, y_train):.3f}")

cm = confusion_matrix(y_test, y_pred_dt)
print("Confusion Matrix:")
print(cm)

Decision Tree Accuracy: 0.8221
Decision Tree Precision: 0.1387
Decision Tree Recall: 0.1170
Decision Tree F1-Score: 0.1269
Tree depth: 20
Number of leaves: 1628
Training accuracy: 0.932
Confusion Matrix:
[[3879  385]
 [ 468   62]]


# Ensemble Methods
### 1. Random Forest

In [176]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

# Train a Random Forest
rf = RandomForestClassifier(
    n_estimators=100,      # number of trees
    max_features='sqrt',   # features per split = sqrt(p)
    oob_score=True,        # compute out-of-bag accuracy
    random_state=42
)
rf.fit(X_train_scaled, y_train)

print(f"OOB accuracy: {rf.oob_score_:.3f}")
print(f"Training accuracy: {rf.score(X_train_scaled, y_train):.3f}")

OOB accuracy: 0.882
Training accuracy: 1.000


### 2. AdaBoost

In [177]:
from sklearn.ensemble import AdaBoostClassifier

# Create and fit AdaBoost
ada = AdaBoostClassifier(n_estimators=50, random_state=42, learning_rate=1.0)
ada.fit(X_train_scaled, y_train)

# Make predictions on test set
y_pred_ada = ada.predict(X_test_scaled)

# Evaluate
print(f"AdaBoost Accuracy: {accuracy_score(y_test, y_pred_ada):.4f}")
print(f"AdaBoost Precision: {precision_score(y_test, y_pred_ada):.4f}")
print(f"AdaBoost Recall: {recall_score(y_test, y_pred_ada):.4f}")
print(f"AdaBoost F1-Score: {f1_score(y_test, y_pred_ada):.4f}")

cm_ada = confusion_matrix(y_test, y_pred_ada)
print("Confusion Matrix:")
print(cm_ada)

AdaBoost Accuracy: 0.8894
AdaBoost Precision: 0.0000
AdaBoost Recall: 0.0000
AdaBoost F1-Score: 0.0000
Confusion Matrix:
[[4264    0]
 [ 530    0]]


c:\Users\micha\OneDrive\Documents1\homework\RSM338\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### 3: XGBoost

In [189]:
from xgboost import XGBClassifier

# Train XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    reg_lambda=1.0,        # L2 regularization
    random_state=42
)
xgb.fit(X_train_scaled, y_train)

print(f"Training accuracy: {xgb.score(X_train_scaled, y_train):.3f}")
print(f"Test accuracy: {xgb.score(X_test_scaled, y_test):.3f}")

Training accuracy: 0.884
Test accuracy: 0.889
